# Semantic pipeline: architecture -- KMeans or UMAP+HDBSCAN?

**Purpose:** decide what the semantic clustering module should actually be built around,
scored by c_v (the metric selected in notebook 4) rather than silhouette. Silhouette is
still reported throughout for visibility and continuity with earlier work, but the
decision here rests on c_v.

**Representations compared, each with a stated reason:**
- **raw** -- isolates whether reduction helps at all.
- **PCA (linear)** -- isolates linear reduction from UMAP's non-linearity specifically.
  PCA, not SVD: centering is the standard, textbook-correct choice for dense sentence
  embeddings (SVD's centering-avoidance only matters for the lexical path's sparse count
  data).
- **UMAP, default params** (`n_neighbors=15, min_dist=0.1`).
- **UMAP, clustering-tuned** (`min_dist=0.0`) -- UMAP's defaults are tuned for
  visualization, not for feeding a density-based clusterer.
- **UMAP, wider neighborhoods** (`n_neighbors=30, min_dist=0.0`).

**Method:** for each representation, the best KMeans fit (sweeping k) and the best HDBSCAN
fit (sweeping `min_cluster_size`), both scored by c_v on the actual cluster members (noise
points excluded for HDBSCAN, same treatment as silhouette).


In [1]:
import json
import sys
import time

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

sys.path.insert(0, ".")
from semantic_diagnostics import embed, build_representations, best_kmeans, best_hdbscan

with open("../data/stackoverflow_sample_2021_03_01_n5000.json") as f:
    data = json.load(f)
titles = pd.Series([row["title"] for row in data]).reset_index(drop=True)
print(f"{len(titles)} titles")

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed(titles, model)

t0 = time.perf_counter()
representations = build_representations(embeddings, n_components=15)
print(f"built {len(representations)} representations in {time.perf_counter() - t0:.1f}s")


/home/olga/Documents/personal/clustering-json-fields/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5000 titles


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12040.50it/s]

/home/olga/Documents/personal/clustering-json-fields/venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/home/olga/Documents/personal/clustering-json-fields/venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/home/olga/Documents/personal/clustering-json-fields/venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


built 5 representations in 44.9s


## The full crossed comparison, scored by c_v

In [2]:
K_VALUES = range(5, 150, 5)
MIN_CLUSTER_SIZE_VALUES = [5, 10, 20, 40, 80]

results = []
hdbscan_detail = {}

for name, rep in representations.items():
    t0 = time.perf_counter()
    km_best = best_kmeans(rep, embeddings, titles, K_VALUES)
    km_time = time.perf_counter() - t0
    results.append({"representation": name, "algorithm": "KMeans", "time_s": round(km_time, 1), **km_best})

    t0 = time.perf_counter()
    hdb_best, hdb_table = best_hdbscan(rep, embeddings, titles, MIN_CLUSTER_SIZE_VALUES)
    hdb_time = time.perf_counter() - t0
    hdbscan_detail[name] = hdb_table
    if hdb_best is not None:
        results.append({"representation": name, "algorithm": "HDBSCAN", "time_s": round(hdb_time, 1), **hdb_best})
    else:
        results.append({"representation": name, "algorithm": "HDBSCAN", "time_s": round(hdb_time, 1),
                         "param": "n/a", "n_clusters": 0, "noise_fraction": 1.0, "c_v": None, "silhouette": None})

    print(f"done: {name} (kmeans {km_time:.1f}s, hdbscan {hdb_time:.1f}s)")

results_df = pd.DataFrame(results)[["representation", "algorithm", "param", "n_clusters", "noise_fraction", "c_v", "silhouette", "time_s"]]
results_df


done: raw (no reduction) (kmeans 204.1s, hdbscan 72.6s)


done: pca (linear) (kmeans 70.1s, hdbscan 3.2s)


done: umap nn=15 min_dist=0.1 (default) (kmeans 69.5s, hdbscan 7.6s)


done: umap nn=15 min_dist=0.0 (clustering-tuned) (kmeans 69.8s, hdbscan 7.9s)


done: umap nn=30 min_dist=0.0 (kmeans 69.5s, hdbscan 7.1s)


,representation,algorithm,param,n_clusters,noise_fraction,c_v,silhouette,time_s
0,raw (no reduction),KMeans,k=5,5,0.000,0.3662,0.0231,204.1
1,raw (no reduction),HDBSCAN,min_cluster_size=10,2,0.968,0.3668,0.1314,72.6
2,pca (linear),KMeans,k=15,15,0.000,0.3656,0.0214,70.1
3,pca (linear),HDBSCAN,min_cluster_size=10,2,0.839,0.3310,0.0110,3.2
4,umap nn=15 min_dist=0.1 (default),KMeans,k=65,65,0.000,0.3510,0.0202,69.5
5,umap nn=15 min_dist=0.1 (default),HDBSCAN,min_cluster_size=40,12,0.548,0.3583,0.0267,7.6
6,umap nn=15 min_dist=0.0 (clustering-tuned),KMeans,k=55,55,0.000,0.3465,0.0200,69.8
7,umap nn=15 min_dist=0.0 (clustering-tuned),HDBSCAN,min_cluster_size=40,16,0.464,0.3685,0.0265,7.9
8,umap nn=30 min_dist=0.0,KMeans,k=5,5,0.000,0.3605,0.0155,69.5
9,umap nn=30 min_dist=0.0,HDBSCAN,min_cluster_size=40,17,0.494,0.3604,0.0301,7.1


## HDBSCAN detail per representation

In [3]:
for name, table in hdbscan_detail.items():
    print(f"=== {name} ===")
    print(table.to_string(index=False))
    print()


=== raw (no reduction) ===
              param  n_clusters  noise_fraction    c_v  silhouette
 min_cluster_size=5           2           0.558 0.1952      0.0141
min_cluster_size=10           2           0.968 0.3668      0.1314
min_cluster_size=20           2           0.975 0.3359      0.1574
min_cluster_size=40           0           1.000    NaN         NaN
min_cluster_size=80           0           1.000    NaN         NaN

=== pca (linear) ===
              param  n_clusters  noise_fraction    c_v  silhouette
 min_cluster_size=5           2           0.417 0.2706     -0.0032
min_cluster_size=10           2           0.839 0.3310      0.0110
min_cluster_size=20           0           1.000    NaN         NaN
min_cluster_size=40           0           1.000    NaN         NaN
min_cluster_size=80           0           1.000    NaN         NaN

=== umap nn=15 min_dist=0.1 (default) ===
              param  n_clusters  noise_fraction    c_v  silhouette
 min_cluster_size=5         152      

## Observations

**The near-total-noise degenerate pattern shows up on raw and PCA both, at different
severities.** Raw's HDBSCAN reaches a high c_v (0.3668) by keeping only 3.2% of the data
(96.8% noise). PCA's version of the same failure mode is milder: its best is only 2
clusters at 83.9% noise, c_v=0.3310. Both are excluded from the real comparison below --
excluding almost everything makes what's left easy to score well on, regardless of how
severe the specific number is.

**At genuinely moderate noise levels (35-55%, the UMAP variants), HDBSCAN's c_v edge over
KMeans is small.** UMAP default (KMeans best 0.3656 vs. HDBSCAN's best 0.3583 at that
representation), UMAP clustering-tuned (0.3685, HDBSCAN's single best result overall),
UMAP wider neighborhoods (0.3604). Comparing the single best result for each algorithm
overall: KMeans on raw or PCA (0.3662 / 0.3656, virtually tied) vs. HDBSCAN on UMAP
clustering-tuned at 46.4% noise (0.3685) -- under 1% apart, while HDBSCAN discards nearly
half the data to get there.

**Reduction gives KMeans a real speed benefit.** Raw took 204.1s to sweep; PCA-reduced
took 70.1s (roughly 3x faster) for virtually identical c_v (0.3662 vs. 0.3656).

## Conclusion

**KMeans on PCA-reduced embeddings is the architecture.** HDBSCAN's real c_v advantage
over KMeans, once the degenerate near-total-noise results are set aside, is under 1% at
best, bought at the cost of excluding roughly half the data -- too small a quality
tradeoff against HDBSCAN's incomplete, non-reviewable clustering for this app's actual use
case (a complete clustering of uploaded data the user can review end to end).

Notebook 6 (shape/optimizability of c_v-based k-selection) can proceed on that basis:
KMeans, scored by c_v, is the settled architecture.
